In [ ]:
from sbi_particle_physics.managers.plotter import Plotter
from sbi_particle_physics.managers.backup import Backup
from sbi_particle_physics.config import DATA_DIR, PLOT_COLORS, MODELS_DIR, DEFAULT_STRIDE, DEFAULT_PRE_N, DEFAULT_PRERUNS, ACCEPTANCE_COEFFS_PATH, C9
import matplotlib as plt
from sbi_particle_physics.managers.improvements import Improvements
import numpy as np
from sbi_particle_physics.managers.imperfections_diagnostics import ImperfectionsDiagnostics
from sbi_particle_physics.objects.model import Model
from sbi_particle_physics.managers.model_diagnostics import ModelDiagnostics

In [ ]:
Plotter.poster_plot()

In [ ]:
plt.rcParams['font.family'] = 'Helvetica' #Helvetica'#'DejaVu Sans'
#plt.rcParams['font.weight'] = 'medium'
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=PLOT_COLORS)
device = "cpu"
paths = Backup.detect_files(DATA_DIR / "data_3")
raw_data, raw_parameters, metadata = Backup.load_data(paths[0:1], device=device)
print(raw_data.shape)
Plotter.plot_a_sample_1D(raw_data[0,:,0], raw_parameters[0], r"$\boldmath{q^2}$ \fontsize{28}{12}\selectfont [GeV$^2$]")

In [ ]:
model = Backup.load_model_for_inference_basic(MODELS_DIR / "training_13", device)


In [ ]:
files = Backup.detect_files(DATA_DIR / "data_3")[-5:] # 5 last files
raw_data, raw_parameters, _ = Backup.load_data(files, model.device)
raw_data = raw_data[:,:model.n_points]
raw_parameters = raw_parameters[:,:model.n_points]
data = model.normalizer.normalize_data(raw_data)
parameters = model.normalizer.normalize_parameters(raw_parameters)

In [ ]:
true_parameter = parameters[0]
observed_sample = data[0]
sampled_parameters = model.draw_parameters_from_predicted_posterior(observed_sample, n_parameters=100000)
sampled_parameters = sampled_parameters.squeeze(0)
print(sampled_parameters.shape)
print(sampled_parameters[:5])
Plotter.plot_a_posterior_parameter(sampled_parameters[:,0], "$C_9$", true_parameter[0].item(), range=(3,5))

In [ ]:
directories_n_points = [MODELS_DIR / f"training_{x}" for x in range(12,22)]
n_posterior_samples = 100
files = Backup.detect_files(DATA_DIR / "data_3")[-1:] # 1 last files
raw_data, raw_parameters, _ = Backup.load_data(files, device)
Improvements.plot_width_by_npoints(directories_n_points, device, raw_data, n_posterior_samples=n_posterior_samples)

In [ ]:
directories_noise = [MODELS_DIR / f"training_12", MODELS_DIR / f"training_28"]
noise_levels = np.linspace(0.0, 0.3, 15).tolist()
Improvements.plot_drift_by_noise_poster(directories_noise, device, raw_data, noise_levels=noise_levels, n_posterior_samples=n_posterior_samples, labels=["Model 1", "Model 2"])

In [ ]:
n_points = 5000
model = Model(device, n_points=n_points)

model.set_prior_basic([3], [5])
model.set_simulator(stride=DEFAULT_STRIDE, pre_N=DEFAULT_PRE_N, preruns=DEFAULT_PRERUNS, use_imperfections=True, acceptance_coeffs_path=ACCEPTANCE_COEFFS_PATH)
model.set_normalizer(model.to_tensor([0,0,0,0,0]), model.to_tensor([1,1,1,1,1]))
model.simulator.imperfections.use_acceptance = False
x_sim_norm = model.simulate_data_with_parameters(model.to_tensor([[C9]]), n_points=n_points).squeeze(0)
model.simulator.imperfections.use_acceptance = True
x_sim = model.normalizer.denormalize_data(x_sim_norm)
x_accnorm = model.simulate_data_with_parameters(model.to_tensor([[C9]]), n_points=n_points).squeeze(0)
x_acc = model.normalizer.denormalize_data(x_accnorm)

In [ ]:
ImperfectionsDiagnostics.compare_simulated_vs_accepted_poster(x_sim=x_sim, x_acc=x_acc, q2_bin=(1.1,6), bins=40)

In [ ]:
model = Backup.load_model_for_inference_basic(directory=MODELS_DIR / "training_12", device=device)
data = model.normalizer.normalize_data(raw_data)
parameters = model.normalizer.normalize_parameters(raw_parameters)
ModelDiagnostics.expected_coverage_test(model, data[:400], parameters[:400], num_posterior_samples=1000)